# paquete-zenodo: archivos del depósito de datos (Zenodo)
Especificación: `claude/encargo_paquete_zenodo.md`. **CPU, Internet OFF.** No entrena ni infiere.

**Inputs (salidas de notebooks):** `datasets-limpios-icdas-segmentacion`, `base-icdas-v4`, `benchmark-icdas-v4`, `benchmark-segmentacion-cv-nino`, `sensibilidad-relectura-v4`, `errores-segmentacion-v3`, `iou-caja-segmentadores`, `pipeline-icdas-v4`. Opcional: dataset `ahmed-full-dataset` (solo para comprobar nombres de archivo y coordenadas; no se copia ninguna foto).

**Salida:** `/kaggle/working/zenodo/`. No contiene fotos ni recortes. Si un chequeo falla, el notebook se detiene.

In [ ]:
import os, glob, zipfile, io, re, json, hashlib
import numpy as np, pandas as pd, cv2
from collections import Counter
from sklearn.metrics import f1_score
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80)
IN, OUT = '/kaggle/input', '/kaggle/working/zenodo'
os.makedirs(OUT, exist_ok=True)
def find(name):
    h = sorted(glob.glob(f'{IN}/**/{name}', recursive=True)); return h[0] if h else None
CL = ['0', 'A', 'B', 'C']
def letter(s): return s.map(lambda v: np.nan if pd.isna(v) else (v if isinstance(v, str) and v in CL else CL[int(float(v))]))
def merged(s): return s.map(lambda v: np.nan if pd.isna(v) else CL[[0, 1, 1, 2, 2, 3, 3][int(v)]])
SEG = {'unetpp_r50': 'U-Net++', 'deeplabv3p_r50': 'DeepLabV3+', 'segformer_b2': 'SegFormer-B2', 'yolo11m_seg': 'YOLO11m-seg'}
GROUP = {'incisivos': 'incisors', 'caninos': 'canines', 'premolares': 'premolars', 'molares 1.º–2.º': 'first and second molars', 'terceros molares': 'third molars'}
RESERVED = ['C0223', 'C0293', 'C0187', 'C0022', 'C0302', 'C0036', 'C0253', 'C0376', 'C0017', 'C0203']   # niños 1–10 del paper
CHECKS = []
def check(name, ok, detail=''):
    CHECKS.append(dict(check=name, ok=bool(ok), detail=str(detail))); print(('OK   ' if ok else 'FALLA'), name, '|', detail)
AH = glob.glob(f'{IN}/**/Dataset/Images', recursive=True); AH = AH[0] if AH else None
print('Fotos de Ahmed disponibles:', AH)

## Etapa 1: máscaras (576 × 576), labelmap y manifiesto
Las máscaras están en el marco de la imagen curada de 576 × 576. `a11…a23` es la matriz afín foto original → 576 × 576. Valor de píxel = número FDI; 99 = diente temporal (colores fuera del labelmap); 0 = fondo.

In [ ]:
man = pd.read_csv(find('segmentation_manifest.csv'))
z = zipfile.ZipFile(find('fdi_segmentation_curated.zip')); names = z.namelist()
lm = [l.strip() for l in z.read([n for n in names if n.endswith('labelmap.txt')][0]).decode('utf-8').splitlines() if l.strip() and not l.startswith('#')]
def to_fdi(u):
    if u <= 8: return 10 + (9 - u)
    if u <= 16: return 20 + (u - 8)
    if u <= 24: return 30 + (25 - u)
    return 40 + (u - 24)
LUT = {}
for l in lm:
    nm, col = l.split(':')[0].strip(), l.split(':')[1]
    if nm.lower() in ('background', 'fondo', 'bg'): continue
    r, g, b = (int(v) for v in col.split(','))
    LUT[(r << 16) | (g << 8) | b] = to_fdi(int(re.search(r'\d+', nm).group()))
assert len(LUT) == 32 and sorted(LUT.values()) == sorted([q * 10 + k for q in (1, 2, 3, 4) for k in range(1, 9)])
def zread(rel, flag):
    return cv2.imdecode(np.frombuffer(z.read([n for n in names if n.endswith(rel)][0]), np.uint8), flag)
PRIMARY = 99
rows, extra, MASKS = [], Counter(), {}
zout = zipfile.ZipFile(f'{OUT}/stage1_masks.zip', 'w', zipfile.ZIP_STORED)
for r in man.itertuples():
    m = zread(r.mask_file, cv2.IMREAD_UNCHANGED)
    assert m.ndim == 3 and m.shape[:2] == (576, 576)
    key = (m[..., 2].astype(np.int32) << 16) | (m[..., 1].astype(np.int32) << 8) | m[..., 0]
    out = np.zeros(key.shape, np.uint8)
    for k in np.unique(key):
        if k == 0: continue
        if k in LUT: out[key == k] = LUT[k]
        else: out[key == k] = PRIMARY; extra[int(k)] += 1
    MASKS[r.image_id] = out
    ok, buf = cv2.imencode('.png', out, [cv2.IMWRITE_PNG_COMPRESSION, 9]); assert ok
    zout.writestr(f'masks/{r.image_id}.png', buf.tobytes())
    for t in np.unique(out):
        if 0 < t < PRIMARY: rows.append((r.image_id, int(t), int((out == t).sum())))
    rows.append((r.image_id, -1, int((out == PRIMARY).sum())))
zout.close()
T = pd.DataFrame(rows, columns=['image_id', 'tooth', 'gt_px'])
prim = T[T.tooth == -1].set_index('image_id').gt_px; T = T[T.tooth > 0]
print('dientes de referencia:', len(T), '| colores fuera del labelmap:', len(extra), '| fotos con temporales:', int((prim > 0).sum()))
check('9710 dientes de referencia en las máscaras convertidas', len(T) == 9710, len(T))
ER = pd.read_csv(find('errores_v3_dientes_referencia.csv.gz')); EP = pd.read_csv(find('errores_v3_clases_predichas.csv.gz'))
e1 = ER[ER.model == 'SegFormer-B2'][['image_id', 'tooth', 'gt_px']].merge(T, on=['image_id', 'tooth'], how='outer', suffixes=('_err', '_mask'))
check('píxeles por diente = tabla de errores (9710 filas, exacto)', len(e1) == 9710 and (e1.gt_px_err == e1.gt_px_mask).all(), int((e1.gt_px_err != e1.gt_px_mask).sum()))
mix = ER.groupby('image_id').mixed.first()
man['mixed_dentition'] = man.image_id.map(mix).astype(int)
man['primary_px'] = man.image_id.map(prim).astype(int)
print('fotos mixtas:', int(man.mixed_dentition.sum()), '| niños:', man[man.mixed_dentition == 1].child_id.nunique(), '| mixtas sin píxeles temporales:', int(((man.mixed_dentition == 1) & (man.primary_px == 0)).sum()), '| no mixtas con temporales:', int(((man.mixed_dentition == 0) & (man.primary_px > 0)).sum()))
check('35 fotos de 10 niños con dentición mixta', man.mixed_dentition.sum() == 35 and man[man.mixed_dentition == 1].child_id.nunique() == 10)
S1 = man.assign(mask_file='masks/' + man.image_id + '.png', n_teeth=man.image_id.map(T.groupby('image_id').size()).astype(int))
S1 = S1[['image_id', 'mask_file', 'child_id', 'fold', 'ahmed_image', 'condition', 'view', 'n_teeth', 'mixed_dentition', 'a11', 'a12', 'a13', 'a21', 'a22', 'a23']]
S1.to_csv(f'{OUT}/stage1_photographs.csv', index=False)
check('630 fotos, 96 niños, 126 por vista, 240 con retractor', len(S1) == 630 and S1.child_id.nunique() == 96 and (S1.view.value_counts() == 126).all() and (S1.condition == 'retractors').sum() == 240)
pd.DataFrame([dict(value=0, fdi='', name='background')] + [dict(value=v, fdi=v, name=f'permanent tooth {v}') for v in sorted(LUT.values())] + [dict(value=PRIMARY, fdi='', name='primary tooth (not numbered)')]).to_csv(f'{OUT}/stage1_labelmap.csv', index=False)

## Control de coordenadas con las fotos de Ahmed (si están)
1) `warpAffine(foto, A)` debe reproducir la imagen curada de 576 × 576. 2) `foto[y0:y1, x0:x1]` debe reproducir el recorte. 3) Todo `ahmed_image` existe.

In [ ]:
MAN2 = None
z2 = zipfile.ZipFile(find('icdas_v4.zip')); n2 = z2.namelist()
M = pd.read_csv(io.BytesIO(z2.read([n for n in n2 if n.endswith('manifest.csv')][0])))
RV = pd.read_csv(find('reserva_por_diente_foto.csv'))
VIEWDIR = {'Frontal': 'Frontal', 'Mandibular occlusal': 'Mandibular', 'Maxillary occlusal': 'Maxillary_Occlusal', 'Left lateral': 'Left_Lateral', 'Right lateral': 'Right_Lateral'}
RV['ahmed_image'] = RV.photo.map(lambda n: 'retractors' if re.match(r'anonymous[_-]\d+[_-]\d+[_-]\d+[_-]0*1[_-]', n) else 'no_retractors') + '/' + RV.view.map(VIEWDIR) + '/' + RV.photo
print('reservados, condición de las fotos:', RV.ahmed_image.str.split('/').str[0].value_counts().to_dict(), '| fotos:', RV.ahmed_image.nunique())
if AH:
    allp = pd.concat([man.ahmed_image, M.ahmed_image, RV.ahmed_image]).drop_duplicates()
    miss = [p for p in allp if not os.path.exists(os.path.join(AH, p))]
    check('todo ahmed_image existe en el dataset de Ahmed', not miss, f'{len(allp)} fotos; faltan {len(miss)} {miss[:3]}')
    rng = np.random.default_rng(0); cors = []
    for i in rng.choice(len(man), 40, replace=False):
        r = man.iloc[i]; ph = cv2.imread(os.path.join(AH, r.ahmed_image), cv2.IMREAD_COLOR)
        A = np.array([[r.a11, r.a12, r.a13], [r.a21, r.a22, r.a23]], np.float64)
        w = cv2.warpAffine(ph, A, (576, 576), flags=cv2.INTER_AREA)
        cu = zread(r.image_file, cv2.IMREAD_COLOR)
        a, b = cv2.cvtColor(w, cv2.COLOR_BGR2GRAY).astype(np.float32).ravel(), cv2.cvtColor(cu, cv2.COLOR_BGR2GRAY).astype(np.float32).ravel()
        cors.append(float(np.corrcoef(a, b)[0, 1]))
    cors = np.array(cors); print('correlación foto transformada vs imagen curada (40 fotos): min %.4f | p10 %.4f | mediana %.4f' % (cors.min(), np.percentile(cors, 10), np.median(cors)))
    check('la matriz afín reproduce la imagen curada (correlación mínima ≥ 0,95)', cors.min() >= 0.95, round(float(cors.min()), 4))
    dif = []
    for i in rng.choice(len(M), 40, replace=False):
        r = M.iloc[i]; ph = cv2.imread(os.path.join(AH, r.ahmed_image), cv2.IMREAD_COLOR)
        c = cv2.imdecode(np.frombuffer(z2.read([n for n in n2 if n.endswith(r.crop_file)][0]), np.uint8), cv2.IMREAD_COLOR)
        x0, y0, x1, y1 = (int(round(v)) for v in (r.x0, r.y0, r.x1, r.y1)); p = ph[y0:y1, x0:x1]
        dif.append(np.inf if p.shape != c.shape else float(np.abs(p.astype(np.int16) - c.astype(np.int16)).mean()))
    print('diferencia media absoluta recorte vs foto[y0:y1, x0:x1] (40 recortes): máx %.3f' % max(dif))
    check('las cajas reproducen los recortes (diferencia media ≤ 1 nivel)', max(dif) <= 1.0, round(max(dif), 3))
else:
    print('AVISO: sin las fotos de Ahmed no se comprueban nombres ni coordenadas.')

## Etapa 2: recortes (cajas y etiquetas), relectura y predicciones fuera de fold

In [ ]:
PRIV = pd.read_csv(find('manifest_privado_v4.csv')); RL = pd.read_csv(find('relectura_cruce_v4.csv'))
assert len(M) == 3466 and M.crop_id.is_unique and RL.crop_v4.is_unique and set(RL.crop_v4) <= set(M.crop_id)
M = M.merge(RL[['crop_v4', 'grupo', 'micdas_original', 'micdas_adjudicado', 'micdas_v4']].rename(columns={'crop_v4': 'crop_id'}), on='crop_id', how='left')
M['micdas'] = M.micdas.astype(str)
check('la etiqueta v4 del cruce de relectura = etiqueta del manifiesto', (M.micdas_v4.dropna().astype(str) == M.micdas[M.micdas_v4.notna()]).all())
M['reread'] = M.grupo.notna().astype(int)
M['reread_group'] = M.grupo.map({'control': 'control', 'senal_moderada': 'moderate_signal', 'senal_fuerte': 'strong_signal'})
M['merged_icdas_before_rereading'] = M.micdas_original.astype(object).where(M.reread == 1, M.micdas).astype(str)
print('releídos:', int(M.reread.sum()), M.reread_group.value_counts().to_dict(), '| etiquetas que cambian:', int((M.merged_icdas_before_rereading != M.micdas).sum()))
print(pd.crosstab(M.merged_icdas_before_rereading[M.reread == 1], M.micdas[M.reread == 1]).to_string())
P2 = PRIV.set_index('crop_id_v3').reindex(M.crop_id)
for c in ('icdas', 'icdas_v2', 'icdas_original'):
    mm = merged(P2[c]).values; rr = (M.reread == 1).values
    print(f'  privado.{c}: nulos {int(P2[c].isna().sum())} | fusionado = etiqueta previa a la relectura en releídos: {int((mm[rr] == M.merged_icdas_before_rereading.values[rr]).sum())} de {int(rr.sum())} | = etiqueta final en todos: {int((mm == M.micdas.values).sum())}')
check('ICDAS del manifiesto privado = ICDAS del manifiesto', (P2.icdas.values == M.icdas.values).all())
M['icdas_before_rereading'] = P2.icdas_v2.values.astype(int)
check('ICDAS previo a la relectura (icdas_v2) coherente con la clase fusionada previa en los 3466 y distinto del final solo en releídos', (merged(M.icdas_before_rereading) == M.merged_icdas_before_rereading).all() and ((M.icdas_before_rereading != M.icdas) & (M.reread == 0)).sum() == 0, int((M.icdas_before_rereading != M.icdas).sum()))
S2 = M.rename(columns={'micdas': 'merged_icdas', 'seg_image_id': 'stage1_image_id'})
S2['in_chained'] = S2.fdi.notna().astype(int)
S2['fdi'] = S2.fdi.astype('Int64')
if all(np.allclose(S2[c], S2[c].round()) for c in ('x0', 'y0', 'x1', 'y1')):
    for c in ('x0', 'y0', 'x1', 'y1'): S2[c] = S2[c].round().astype(int)
else: print('AVISO: cajas con decimales; se dejan como están')
S2 = S2[['crop_id', 'child_id', 'fold', 'ahmed_image', 'condition', 'view', 'x0', 'y0', 'x1', 'y1', 'icdas', 'merged_icdas', 'icdas_before_rereading', 'merged_icdas_before_rereading', 'reread', 'reread_group', 'stage1_image_id', 'fdi', 'in_chained']]
S2.to_csv(f'{OUT}/stage2_crops.csv', index=False)
vc = S2.merged_icdas.value_counts().to_dict()
check('3466 recortes, 954 fotos, 411 niños; 871 / 1267 / 879 / 449', len(S2) == 3466 and S2.ahmed_image.nunique() == 954 and S2.child_id.nunique() == 411 and [vc[c] for c in CL] == [871, 1267, 879, 449], vc)
check('510 recortes releídos (293 / 157 / 60)', S2.reread.sum() == 510 and S2.reread_group.value_counts().to_dict() == {'moderate_signal': 293, 'strong_signal': 157, 'control': 60})

OOF = pd.read_csv(find('oof_predicciones.csv'))
assert len(OOF) == 7 * 3466 and set(OOF.crop_id) == set(S2.crop_id)
o = OOF.merge(S2[['crop_id', 'child_id', 'fold', 'merged_icdas', 'merged_icdas_before_rereading']], on='crop_id', suffixes=('', '_m'))
check('niño, fold y etiqueta de las predicciones = manifiesto', (o.child_id == o.child_id_m).all() and (o.fold == o.fold_m).all() and (letter(o.y) == o.merged_icdas).all())
o['pred'] = letter(o.pred)
O2 = o[['model', 'crop_id', 'child_id', 'fold', 'merged_icdas', 'pred', 'p_0', 'p_A', 'p_B', 'p_C']].sort_values(['model', 'crop_id'])
O2.to_csv(f'{OUT}/stage2_oof_predictions.csv', index=False, float_format='%.6f')
mf = {m: f1_score(d.merged_icdas, d.pred, average='macro') for m, d in O2.groupby('model')}
print({k: round(v, 4) for k, v in mf.items()})
check('macro F1 de Swin-T = 0,838; rango 0,811–0,838', abs(mf['swin_tiny'] - 0.838) < 0.0006 and abs(min(mf.values()) - 0.811) < 0.0006 and max(mf, key=mf.get) == 'swin_tiny', round(mf['swin_tiny'], 4))
sw = o[o.model == 'swin_tiny']; f_or = f1_score(sw.merged_icdas_before_rereading, sw.pred, average='macro')
check('macro F1 de Swin-T con las etiquetas previas a la relectura = 0,794', abs(f_or - 0.794) < 0.0006, round(f_or, 4))

## Etapa 1: tablas por diente (fuera de fold)

In [ ]:
check('tabla de errores: 9710 dientes × 4 modelos; SegFormer-B2 identifica 93,6 %', len(ER) == 38840 and abs((ER[ER.model == 'SegFormer-B2'].category == 'identified').mean() - 0.9364) < 0.0001, round((ER[ER.model == 'SegFormer-B2'].category == 'identified').mean(), 4))
for d in (ER, EP): d['mixed'] = d.mixed.astype(int)
ER['pred_tooth'] = ER.pred_tooth.astype('Int64')
ER.to_csv(f'{OUT}/stage1_reference_teeth.csv.gz', index=False, float_format='%.6f')
EP.to_csv(f'{OUT}/stage1_predicted_teeth.csv.gz', index=False, float_format='%.6f')

BX = pd.read_csv(find('iou_caja_por_diente.csv'), low_memory=False)
assert len(BX) == 38840
BX['model'] = BX.model.map(SEG); BX['tooth_group'] = BX.grupo.map(GROUP); assert BX.model.notna().all() and BX.tooth_group.notna().all()
ren = {'fdi': 'tooth', 'gy0': 'ref_y0', 'gx0': 'ref_x0', 'gy1': 'ref_y1', 'gx1': 'ref_x1', 'gt_area_caja': 'ref_box_area', 'py0': 'pred_y0', 'px0': 'pred_x0', 'py1': 'pred_y1', 'px1': 'pred_x1',
       'iou': 'box_iou', 'razon_area': 'box_area_ratio', 'n_comp': 'n_components', 'ly0': 'lcc_y0', 'lx0': 'lcc_x0', 'ly1': 'lcc_y1', 'lx1': 'lcc_x1', 'iou_S1': 'box_iou_lcc', 'razon_area_S1': 'box_area_ratio_lcc',
       'found_S2': 'found_raw', 'iou_S2': 'box_iou_raw', 'razon_area_S2': 'box_area_ratio_raw', 'found_S2b': 'found_raw_no_threshold', 'iou_S2b': 'box_iou_raw_no_threshold', 'razon_area_S2b': 'box_area_ratio_raw_no_threshold'}
BX = BX.rename(columns=ren)
BX['found'] = BX.found.astype(int)
for c in ('pred_y0', 'pred_x0', 'pred_y1', 'pred_x1', 'lcc_y0', 'lcc_x0', 'lcc_y1', 'lcc_x1', 'n_components'): BX[c] = BX[c].astype('Int64')
for c in ('found_raw', 'found_raw_no_threshold'): BX[c] = BX[c].map(lambda v: np.nan if pd.isna(v) else int(v in (True, 'True', 1, 1.0))).astype('Int64')
BX = BX[['model', 'image_id', 'child_id', 'fold', 'view', 'condition', 'tooth', 'tooth_group', 'gt_px', 'ref_y0', 'ref_x0', 'ref_y1', 'ref_x1', 'ref_box_area', 'pred_px', 'found'] + [c for c in ren.values() if c.startswith(('pred_', 'box_', 'n_', 'lcc_', 'found_'))]]
mb = BX.assign(b=BX.box_iou.fillna(0)).groupby('model').b.mean()
check('IoU de caja medio: YOLO11m-seg 0,898 y SegFormer-B2 0,872', abs(mb['YOLO11m-seg'] - 0.898) < 0.0006 and abs(mb['SegFormer-B2'] - 0.872) < 0.0006, mb.round(4).to_dict())
BX.to_csv(f'{OUT}/stage1_box_iou.csv.gz', index=False, float_format='%.6f')

DT = pd.read_csv(find('dientes_por_imagen.csv'))
DT['model'] = DT.model.map(SEG); assert DT.model.notna().all()
g = DT[DT.in_gt]
pc = DT.assign(s=DT.gt_px + DT.pred_px).groupby(['model', 'tooth']).agg(I=('inter', 'sum'), S=('s', 'sum'))
dz = (2 * pc.I / pc.S).groupby('model').mean(); iz = (pc.I / (pc.S - pc.I)).groupby('model').mean()
print('Dice de la Table 2 (Dice agrupado por clase FDI sobre todas las fotos, media de 32 clases):', dz.round(4).to_dict(), '| IoU:', iz.round(4).to_dict(), '| filas por modelo:', DT.groupby('model').size().to_dict())
print('otras agregaciones (informativo) | media de los Dice por diente:', g.groupby('model').dice.mean().round(4).to_dict(), '| media de clases de los Dice por diente:', g.groupby(['model', 'tooth']).dice.mean().groupby('model').mean().round(4).to_dict())
EXP_D = {'SegFormer-B2': 0.9013, 'DeepLabV3+': 0.8821, 'U-Net++': 0.8733, 'YOLO11m-seg': 0.8706}; EXP_I = {'SegFormer-B2': 0.835, 'DeepLabV3+': 0.811, 'YOLO11m-seg': 0.808, 'U-Net++': 0.800}
check('Dice e IoU de la Table 2 desde la tabla por diente: 0,901 / 0,882 / 0,873 / 0,871 y 0,835 / 0,811 / 0,800 / 0,808', all(abs(dz[m] - v) < 0.0006 for m, v in EXP_D.items()) and all(abs(iz[m] - v) < 0.0011 for m, v in EXP_I.items()), {m: (round(dz[m], 4), round(iz[m], 4)) for m in EXP_D})
j = g.merge(ER[['model', 'image_id', 'tooth', 'iou', 'gt_px']], on=['model', 'image_id', 'tooth'], suffixes=('', '_err'))
print('IoU por diente del benchmark vs tabla de errores: filas', len(j), '| |dif| > 0,001:', int(((j.iou - j.iou_err).abs() > 0.001).sum()), '| gt_px distinto:', int((j.gt_px != j.gt_px_err).sum()))
check('tabla por diente del benchmark: 9710 dientes de referencia × 4 modelos, mismos píxeles de referencia que la tabla de errores', int(DT.in_gt.sum()) == 38840 and len(j) == 38840 and (j.gt_px == j.gt_px_err).all(), len(j))
for c in ('in_gt', 'in_pred', 'identified', 'tp_pred', 'covered', 'numbered_ok'): DT[c] = DT[c].map(lambda v: np.nan if pd.isna(v) else int(v in (True, 'True', 1, 1.0))).astype('Int64')
DT = DT.rename(columns={'identified': 'identified_dice05', 'tp_pred': 'tp_pred_dice05'}).drop(columns=['cls'])
DT = DT[['model', 'image_id', 'child_id', 'fold', 'view', 'condition', 'tooth', 'in_gt', 'in_pred', 'inter', 'gt_px', 'pred_px', 'dice', 'iou', 'identified_dice05', 'tp_pred_dice05', 'covered', 'numbered_ok', 'hd95', 'fg_dice']]
DT.to_csv(f'{OUT}/stage1_per_tooth_metrics.csv.gz', index=False, float_format='%.6f')

## Evaluación encadenada y niños reservados

In [ ]:
PC = pd.read_csv(find('pipeline_por_recorte.csv'))
assert len(PC) == 908 and set(PC.crop_id) == set(S2[S2.in_chained == 1].crop_id)
y = PC.y.astype(int)
fM, fG = f1_score(y, PC.pred_M.astype(int), average='macro'), f1_score(y, PC.pred_gt.astype(int), average='macro')
fP, fL = f1_score(y, PC.pred_pfdi.fillna(0).astype(int), average='macro'), f1_score(y, PC.pred_ploc.astype(int), average='macro')
check('encadenada: 908 recortes, 68 niños, 166 fotos; macro F1 M 0,778, G 0,773, P-FDI 0,759, P-ubic. 0,770', PC.child_id.nunique() == 68 and PC.ahmed_image.nunique() == 166 and abs(fM - 0.778) < 0.0006 and abs(fG - 0.773) < 0.0006 and abs(fP - 0.759) < 0.0006 and abs(fL - 0.770) < 0.0006, [round(v, 4) for v in (fM, fG, fP, fL)])
check('encadenada: 904 de 908 dientes encontrados con su número (99,6 %)', int(PC.pred_pfdi.notna().sum()) == 904, int(PC.pred_pfdi.notna().sum()))
sb = BX[BX.model == 'SegFormer-B2'][['image_id', 'tooth', 'ref_y0', 'ref_x0', 'ref_y1', 'ref_x1', 'pred_y0', 'pred_x0', 'pred_y1', 'pred_x1']].rename(columns={'image_id': 'stage1_image_id', 'tooth': 'fdi'})
sb.columns = ['stage1_image_id', 'fdi'] + ['mask576_' + c for c in sb.columns[2:]]
CH = PC.rename(columns={'seg_image_id': 'stage1_image_id', 'micdas': 'merged_icdas', 'pred_gt': 'pred_G', 'pred_pfdi': 'pred_PFDI', 'pred_ploc': 'pred_PLOC', 'ploc_fdi': 'PLOC_fdi', 'ploc_cover': 'PLOC_coverage',
                        'iou_gt_manual': 'box_iou_G_vs_M', 'iou_pfdi_manual': 'box_iou_PFDI_vs_M', 'iou_ploc_manual': 'box_iou_PLOC_vs_M'})
CH = CH.rename(columns={f'p_{a}_{c}': f'p_{b}_{c}' for a, b in (('gt', 'G'), ('pfdi', 'PFDI'), ('ploc', 'PLOC')) for c in CL})
for c in ('pred_M', 'pred_G', 'pred_PFDI', 'pred_PLOC'): CH[c] = letter(CH[c])
CH['fdi'] = CH.fdi.astype(int); CH['PLOC_fdi'] = CH.PLOC_fdi.astype('Int64')
CH = CH.merge(sb, on=['stage1_image_id', 'fdi'], how='left'); assert len(CH) == 908
print('recortes de la encadenada sin caja de máscara de referencia:', int(CH.mask576_ref_y0.isna().sum()))
sw = O2[O2.model == 'swin_tiny'].set_index('crop_id').reindex(CH.crop_id)
check('pred_M de la encadenada = predicción fuera de fold de Swin-T', (sw.pred.values == CH.pred_M.values).all(), int((sw.pred.values != CH.pred_M.values).sum()))
for c in CL: CH[f'p_M_{c}'] = sw[f'p_{c}'].values
for c in ('x0', 'y0', 'x1', 'y1'): CH[c] = S2.set_index('crop_id')[c].reindex(CH.crop_id).values
CH = CH[['crop_id', 'child_id', 'fold', 'ahmed_image', 'condition', 'view', 'x0', 'y0', 'x1', 'y1', 'stage1_image_id', 'fdi', 'icdas', 'merged_icdas',
         'pred_M', 'p_M_0', 'p_M_A', 'p_M_B', 'p_M_C', 'pred_G', 'p_G_0', 'p_G_A', 'p_G_B', 'p_G_C', 'pred_PFDI', 'p_PFDI_0', 'p_PFDI_A', 'p_PFDI_B', 'p_PFDI_C',
         'pred_PLOC', 'p_PLOC_0', 'p_PLOC_A', 'p_PLOC_B', 'p_PLOC_C', 'PLOC_fdi', 'PLOC_coverage', 'box_iou_G_vs_M', 'box_iou_PFDI_vs_M', 'box_iou_PLOC_vs_M'] + [c for c in CH.columns if c.startswith('mask576_')]]
CH.to_csv(f'{OUT}/chained_evaluation.csv', index=False, float_format='%.6f')

assert set(RV.child_id) == set(RESERVED)
RV['child'] = RV.child_id.map({c: i + 1 for i, c in enumerate(RESERVED)})
RV['view'] = RV.view.map(VIEWDIR); RV['folds_agree'] = RV.folds_agree.astype(int)
for c in ['pred'] + [f'f{k}_pred' for k in range(5)]: RV[c] = letter(RV[c])
od = pd.read_csv(find('odontogramas_reserva.csv'))
check('reservados: 746 detecciones diente-foto, 279 dientes, 10 niños fuera de ambas bases', len(RV) == 746 and len(od) == 279 and RV[['child_id', 'fdi']].drop_duplicates().shape[0] == 279 and not (set(RESERVED) & (set(S1.child_id) | set(S2.child_id))))
RVo = RV.rename(columns={'area576': 'mask_px_576', 'pmax': 'p_max', 'n_fold_votes_for_pred': 'n_folds_for_pred'})
RVo = RVo[['child', 'child_id', 'ahmed_image', 'view', 'fdi', 'mask_px_576', 'x0', 'y0', 'x1', 'y1', 'pred', 'p_max', 'p_0', 'p_A', 'p_B', 'p_C', 'folds_agree', 'n_folds_for_pred'] + [f'f{k}_pred' for k in range(5)]].sort_values(['child', 'ahmed_image', 'fdi'])
RVo.to_csv(f'{OUT}/reserved_children_teeth.csv', index=False, float_format='%.6f')
od['child'] = od.child_id.map({c: i + 1 for i, c in enumerate(RESERVED)}); od['pred'] = letter(od.clase)
print('odontograma:', od.pred.value_counts().to_dict())
od[['child', 'child_id', 'fdi', 'pred']].to_csv(f'{OUT}/reserved_children_odontograms.csv', index=False)
check('odontogramas: 164 / 108 / 7 / 0', [int((od.pred.astype(str) == c).sum()) for c in CL] == [164, 108, 7, 0], od.pred.value_counts().to_dict())

## Niños del estudio y folds

In [ ]:
a = S1.groupby('child_id').agg(fold1=('fold', 'first'), nf1=('fold', 'nunique'), n_stage1_photographs=('image_id', 'size'), mixed_dentition=('mixed_dentition', 'max'))
b = S2.groupby('child_id').agg(fold2=('fold', 'first'), nf2=('fold', 'nunique'), n_stage2_crops=('crop_id', 'size'), n_stage2_photographs=('ahmed_image', 'nunique'))
K = a.join(b, how='outer')
check('un solo fold por niño en cada etapa', (K.nf1.dropna() == 1).all() and (K.nf2.dropna() == 1).all())
both = K.fold1.notna() & K.fold2.notna()
check('81 niños en ambas bases, con el mismo fold', int(both.sum()) == 81 and (K.fold1[both] == K.fold2[both]).all(), int(both.sum()))
K['fold'] = K.fold1.fillna(K.fold2).astype('Int64')
K['in_stage1'], K['in_stage2'] = K.fold1.notna().astype(int), K.fold2.notna().astype(int)
K['in_chained'] = K.index.isin(CH.child_id).astype(int); K['reserved'] = 0
R = pd.DataFrame(dict(child_id=RESERVED, reserved=1, in_stage1=0, in_stage2=0, in_chained=0)).set_index('child_id')
K = pd.concat([K, R]).reset_index().rename(columns={'index': 'child_id'})
K['reserved_child_number'] = K.child_id.map({c: i + 1 for i, c in enumerate(RESERVED)}).astype('Int64')
for c in ('n_stage1_photographs', 'n_stage2_crops', 'n_stage2_photographs', 'mixed_dentition'): K[c] = K[c].astype('Int64')
K['fold'] = K.fold.astype('Int64')
K = K[['child_id', 'fold', 'in_stage1', 'in_stage2', 'in_chained', 'reserved', 'reserved_child_number', 'n_stage1_photographs', 'mixed_dentition', 'n_stage2_crops', 'n_stage2_photographs']].sort_values('child_id')
K.to_csv(f'{OUT}/children.csv', index=False)
print(len(K), 'niños |', K[['in_stage1', 'in_stage2', 'in_chained', 'reserved']].sum().to_dict())
check('niños: 96 etapa 1, 411 etapa 2, 68 encadenada, 10 reservados; 436 en total', K.in_stage1.sum() == 96 and K.in_stage2.sum() == 411 and K.in_chained.sum() == 68 and K.reserved.sum() == 10 and len(K) == 436 and K.child_id.is_unique, len(K))

## Privacidad, columnas y tamaños

In [ ]:
files = sorted(f for f in os.listdir(OUT) if not f.startswith(('SHA256', 'checks', 'columns')))
BAD = re.compile(r'(/kaggle/|[A-Za-z]:\\|Descargas|Downloads|leonardoaranguiz|odontify|\bLeo\b|\bEdo\b|Daniela|Fabi[aá]n|Ra[uú]l|@|\b[0-9a-f]{32}\b|icdas_\d{5}\b|cand_|minado|reviewer)', re.I)
cols, hits = {}, {}
for f in files:
    if f.endswith('.zip'): continue
    d = pd.read_csv(f'{OUT}/{f}', low_memory=False); cols[f] = dict(rows=len(d), columns=list(d.columns))
    txt = open(f'{OUT}/{f}', 'rb').read() if not f.endswith('.gz') else __import__('gzip').open(f'{OUT}/{f}', 'rb').read()
    h = Counter(m.group(0) for m in BAD.finditer(txt.decode('utf-8')))
    if h: hits[f] = dict(h.most_common(5))
    print(f'\n{f} | {len(d)} filas'); print('  ', list(d.columns))
check('ningún CSV tiene rutas, nombres, correos, md5 ni identificadores antiguos', not hits, hits)
zc = zipfile.ZipFile(f'{OUT}/stage1_masks.zip'); zn = zc.namelist(); vals = set()
for n in zn[::9]:
    raw = zc.read(n); m = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_UNCHANGED); vals |= set(np.unique(m).tolist())
    assert m.ndim == 2 and m.shape == (576, 576) and b'tEXt' not in raw and b'eXIf' not in raw and b'iTXt' not in raw
allowed = {0, PRIMARY} | set(LUT.values())
check('el zip solo tiene 630 máscaras de 1 canal, sin metadatos y con valores 0, FDI y 99', len(zn) == 630 and all(n.startswith('masks/') and n.endswith('.png') for n in zn) and vals <= allowed, sorted(vals)[:6])
imgs = [f for f in os.listdir(OUT) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.pt', '.npz'))]
check('ninguna foto, recorte ni peso suelto en la salida', not imgs, imgs)
sha = []
for f in files:
    hsh = hashlib.sha256(open(f'{OUT}/{f}', 'rb').read()).hexdigest(); sha.append(f'{hsh}  {f}')
    print(f'{os.path.getsize(f"{OUT}/{f}") / 1024:10.0f} KB  {f}')
open(f'{OUT}/SHA256SUMS.txt', 'w').write('\n'.join(sha) + '\n')
print('total: %.1f MB' % (sum(os.path.getsize(f'{OUT}/{f}') for f in os.listdir(OUT)) / 2**20))
json.dump(cols, open('/kaggle/working/columns.json', 'w'), indent=1)
CK = pd.DataFrame(CHECKS); CK.to_csv('/kaggle/working/checks.csv', index=False)
print('\n== CHEQUEOS =='); print(CK.to_string(index=False))
assert CK.ok.all(), 'HAY CHEQUEOS QUE FALLAN: no publicar'
print('TODOS LOS CHEQUEOS OK:', len(CK))